# Exercise 1: Web Server Access Log Analysis
**Student:** Solis (`TUPM-26-2631`)

Processes a student-specific number of web requests from `weblog.csv`, identifies HTTP 3xx responses, exports filtered records, summarizes findings in JSON, and creates a verified backup.


In [1]:
# Student Information & Parameter Derivation
LAST_NAME = "Solis"
STUDENT_ID = "TUPM-26-2631"
SEED_DIGIT = int(STUDENT_ID[-1])  # 1
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())  # 20

RECORDS_LIMIT = 1000 + ID_SUM  # 1020
TARGET_CLASS_DIGIT = (SEED_DIGIT % 4) + 2  # 3 -> 3xx
TARGET_STATUS_CLASS = f"{TARGET_CLASS_DIGIT}xx"

print(f"Student: {LAST_NAME} ({STUDENT_ID})")
print(f"Seed Digit: {SEED_DIGIT}")
print(f"ID Sum: {ID_SUM}")
print(f"Records to Process: {RECORDS_LIMIT}")
print(f"Target Status Class: {TARGET_STATUS_CLASS}")


Student: Solis (TUPM-26-2631)
Seed Digit: 1
ID Sum: 20
Records to Process: 1020
Target Status Class: 3xx


### File & Directory Setup


In [2]:
import os
import csv
import json
import shutil
from collections import Counter
from pathlib import Path

# Setup file paths
BASE_DIR = Path(".")
DATASET_PATH = BASE_DIR / "datasets" / "weblog.csv"
if not DATASET_PATH.exists():
    DATASET_PATH = Path("datasets/weblog.csv")

OUTPUT_DIR = BASE_DIR / "output"
BACKUP_DIR = BASE_DIR / "backup"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

FILTERED_CSV_PATH = OUTPUT_DIR / "weblog_3xx_filtered.csv"
JSON_SUMMARY_PATH = OUTPUT_DIR / "weblog_3xx_summary.json"
BACKUP_JSON_PATH = BACKUP_DIR / "weblog_3xx_summary_backup.json"

print("Dataset Path:", DATASET_PATH.resolve())
print("Output Directory:", OUTPUT_DIR.resolve())
print("Backup Directory:", BACKUP_DIR.resolve())


Dataset Path: C:\Users\Leandro\Downloads\Lab 6\Lab6_File_Handling_Solis\Lab6_File_Handling_Additional_Exercises\datasets\weblog.csv
Output Directory: C:\Users\Leandro\Downloads\Lab 6\Lab6_File_Handling_Solis\Lab6_File_Handling_Additional_Exercises\output
Backup Directory: C:\Users\Leandro\Downloads\Lab 6\Lab6_File_Handling_Solis\Lab6_File_Handling_Additional_Exercises\backup


### Line-by-Line Processing, Filtering, Export, and Assessment


In [3]:
# Process CSV line-by-line without loading entire dataset into memory
records_processed = 0
records_in_target = 0
invalid_records = 0
status_counts = Counter()
url_counts = Counter()
selected_records = []

with open(DATASET_PATH, mode="r", encoding="utf-8", errors="ignore") as f:
    reader = csv.DictReader(f)
    for row in reader:
        if records_processed >= RECORDS_LIMIT:
            break
        records_processed += 1

        ip = (row.get("IP") or "").strip()
        time = (row.get("Time") or "").strip()
        url = (row.get("URL") or "").strip()
        status = (row.get("Staus") or row.get("Status") or "").strip()

        if not status or not status.isdigit():
            invalid_records += 1
            continue

        if status.startswith(str(TARGET_CLASS_DIGIT)) and len(status) == 3:
            records_in_target += 1
            status_counts[status] += 1
            url_counts[url] += 1
            selected_records.append({
                "IP": ip,
                "Time": time,
                "URL": url,
                "Status": status
            })

top_5_urls = url_counts.most_common(5)

# Export selected records to filtered CSV
fieldnames = ["IP", "Time", "URL", "Status"]
with open(FILTERED_CSV_PATH, mode="w", encoding="utf-8", newline="") as csv_out:
    writer = csv.DictWriter(csv_out, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(selected_records)

# Export summary to JSON
summary_data = {
    "assessment_data": {
        "student_name": LAST_NAME,
        "student_id": STUDENT_ID,
        "seed_digit": SEED_DIGIT,
        "id_sum": ID_SUM,
        "records_processed": records_processed,
        "target_status_class": TARGET_STATUS_CLASS,
        "records_in_target_class": records_in_target,
        "invalid_or_skipped_records": invalid_records,
        "count_per_status_code": dict(status_counts),
        "five_most_requested_urls": [
            {"rank": i + 1, "url": u, "count": c}
            for i, (u, c) in enumerate(top_5_urls)
        ],
        "filtered_csv_file": str(FILTERED_CSV_PATH.name),
        "json_summary_file": str(JSON_SUMMARY_PATH.name)
    }
}

with open(JSON_SUMMARY_PATH, mode="w", encoding="utf-8") as json_out:
    json.dump(summary_data, json_out, indent=4)

# Create backup and verify
shutil.copy2(JSON_SUMMARY_PATH, BACKUP_JSON_PATH)
backup_verified = BACKUP_JSON_PATH.exists() and BACKUP_JSON_PATH.stat().st_size > 0
backup_size = BACKUP_JSON_PATH.stat().st_size if backup_verified else 0

# Display Assessment Data Table
print("=" * 70)
print(f"{'Item':<35} | {'Result'}")
print("-" * 70)
print(f"{'Records processed':<35} | {records_processed}")
print(f"{'Target status class':<35} | {TARGET_STATUS_CLASS}")
print(f"{'Records in target class':<35} | {records_in_target}")
status_str = ", ".join(f"{k}: {v}" for k, v in sorted(status_counts.items()))
print(f"{'Count per status code':<35} | {status_str}")
print(f"{'Five most requested URLs':<35} |")
for i, (u, c) in enumerate(top_5_urls, 1):
    print(f"  {i}. {u:<50} ({c} requests)")
print(f"{'Filtered CSV file':<35} | {FILTERED_CSV_PATH.name} ({len(selected_records)} rows)")
print(f"{'JSON summary file':<35} | {JSON_SUMMARY_PATH.name}")
print(f"{'Backup verification':<35} | {'VERIFIED' if backup_verified else 'FAILED'} ({BACKUP_JSON_PATH.name}, {backup_size} bytes)")
print("=" * 70)
print("Final Output: Exercise 1 completed successfully.")


Item                                | Result
----------------------------------------------------------------------
Records processed                   | 1020
Target status class                 | 3xx
Records in target class             | 164
Count per status code               | 302: 143, 304: 21
Five most requested URLs            |
  1. GET / HTTP/1.1                                     (45 requests)
  2. POST /process.php HTTP/1.1                         (35 requests)
  3. GET /contestproblem.php?name=RUET%20OJ%20Server%20Testing%20Contest HTTP/1.1 (34 requests)
  4. GET /home.php HTTP/1.1                             (9 requests)
  5. GET /logout.php HTTP/1.1                           (7 requests)
Filtered CSV file                   | weblog_3xx_filtered.csv (164 rows)
JSON summary file                   | weblog_3xx_summary.json
Backup verification                 | VERIFIED (weblog_3xx_summary_backup.json, 1316 bytes)
Final Output: Exercise 1 completed successfully.
